# Predicting 30-Day Hospital Readmission for Diabetes Patients

**Data Science assignment — end-to-end machine learning project**

| | |
|---|---|
| **Dataset** | Diabetes 130-US Hospitals for years 1999–2008 (UCI ML Repository, ID 296) |
| **Size** | 101,766 encounters × 50 raw columns ✔ (requirement: >100,000 rows, ≥35 columns) |
| **Task** | Binary classification — will the patient be re-admitted within 30 days? |
| **Models** | Logistic Regression · Random Forest · XGBoost · Keras neural network (+ SMOTE comparison) |

Hospital readmission shortly after discharge is expensive and often preventable. US hospitals are financially penalised for excess 30-day readmissions, so flagging high-risk diabetes patients at discharge time lets care teams target follow-up visits. 
The dataset covers 10 years of clinical encounters at 130 US hospitals and was curated by Strack et al. (2014). It satisfies the assignment requirements with **101,766 rows and 50 columns**.

## 1 · Setup

In [1]:
import sys, os
from pathlib import Path

# The notebook lives in notebooks/ — switch to the repo root and import the
# same src/ modules the CLI scripts use (single source of truth).
if not Path('src').exists():
    os.chdir('..')
sys.path.insert(0, os.getcwd())

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn, xgboost, tensorflow as tf

from src.preprocess import get_dataset, load_raw, clean_basic, build_features
import src.eda as eda
import src.evaluate as ev
from src.train import train_all_models

sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams.update({'figure.dpi': 100, 'savefig.bbox': 'tight'})
pd.set_option('display.max_columns', 60)

print('pandas ', pd.__version__)
print('sklearn', sklearn.__version__)
print('xgboost', xgboost.__version__)
print('tensorflow', tf.__version__)

I0000 00:00:1789495989.082190    9741 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1789495989.128965    9741 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


I0000 00:00:1789495990.678419    9741 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


pandas  3.0.5
sklearn 1.9.1
xgboost 3.2.0
tensorflow 2.21.0


## 2 · Load the data

In [2]:
raw = load_raw()
print(f'{raw.shape[0]:,} encounters x {raw.shape[1]} columns')
raw.head()

101,766 encounters x 50 columns


,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,medical_specialty,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,diag_1,diag_2,diag_3,number_diagnoses,max_glu_serum,A1Cresult,metformin,repaglinide,nateglinide,chlorpropamide,glimepiride,acetohexamide,glipizide,glyburide,tolbutamide,pioglitazone,rosiglitazone,acarbose,miglitol,troglitazone,tolazamide,examide,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),NaN,6,25,1,1,NaN,Pediatrics-Endocrinology,41,0,1,0,0,0,250.83,NaN,NaN,1,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),NaN,1,1,7,3,NaN,NaN,59,0,18,0,0,0,276,250.01,255,9,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),NaN,1,1,7,2,NaN,NaN,11,5,13,2,0,1,648,250,V27,6,NaN,NaN,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),NaN,1,1,7,2,NaN,NaN,44,1,16,0,0,0,8,250.43,403,7,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),NaN,1,1,7,1,NaN,NaN,51,0,8,0,0,0,197,157,250,5,NaN,NaN,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,Ch,Yes,NO


In [3]:
# The dataset uses '?' as its missing-value code — replaced with NaN on load.
raw.isna().sum().sort_values(ascending=False).head(10)

weight               98569
max_glu_serum        96420
A1Cresult            84748
medical_specialty    49949
payer_code           40256
race                  2273
diag_3                1423
diag_2                 358
diag_1                  21
patient_nbr              0
dtype: int64

## 3 · Exploratory Data Analysis

### 3.1 The target is heavily imbalanced

In [4]:
fig = eda.plot_target_distribution(raw)
plt.show()

In [5]:
fig = eda.plot_missing_values(raw)
plt.show()

**Observations**
- Only **11.2%** of encounters end in a readmission within 30 days → accuracy is a misleading metric; we will optimise/report **ROC-AUC and PR-AUC**.
- `weight` is missing for 97% of encounters → dropped. `payer_code` (40%) and `medical_specialty` (49%) are kept/patched as an explicit `Missing`/`Other` category.

### 3.2 Numeric feature distributions

In [6]:
data = get_dataset()          # cleaned frame + stratified 80/20 split
df = data['clean']            # readable version for EDA
fig = eda.plot_numeric_distributions(df)
plt.show()

### 3.3 What actually drives readmission

In [7]:
fig = eda.plot_readmission_by_key_features(df)
plt.show()

**Observations** — the early-readmission rate
- rises with **age**;
- is *highest for the shortest stays* (1–3 days — patients discharged perhaps too early);
- explodes with **prior-year utilisation** (outpatient + emergency + inpatient visits), confirming that history is the strongest signal.

In [8]:
fig = eda.plot_correlation_heatmap(df)
plt.show()

In [9]:
fig = eda.plot_categorical_target(df)
plt.tight_layout(); plt.show()

In [10]:
fig = eda.plot_medication_usage(df)
plt.show()

## 4 · Preprocessing & feature engineering

Applied by `src/preprocess.py`:

| Step | Detail |
|---|---|
| Drop | `encounter_id`, `patient_nbr` (identifiers), `weight` (97% missing), `payer_code` (40% missing, weak), `citoglipton`/`examide` (constant) |
| Decode | `admission/discharge/source IDs` → clinical categories via `IDs_mapping.csv`; ~700 ICD-9 codes → 17 disease groups |
| Encode | age brackets → midpoints; 21 medication columns → ordinal {No, Down, Steady, Up}; HbA1c/glucose → ordinal severity |
| Engineer | `service_utilization` (prior visits), `n_active_medications`, `lab_monitoring_score` |
| Target | `readmitted` → binary: `<30` = 1, else 0 |

All **101,766 encounters are kept** (the assignment requires >100,000 rows), including hospice/expired discharges — the model can learn these are never readmitted.

In [11]:
X, y = build_features(data['clean'])
print(f'features: {X.shape[1]}  |  rows: {X.shape[0]:,}  |  positive rate: {y.mean():.2%}')
X.head()

features: 21  |  rows: 101,766  |  positive rate: 11.16%


,age,time_in_hospital,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,number_diagnoses,service_utilization,n_active_medications,lab_monitoring_score,race,gender,medical_specialty,admission_type,discharge_disposition,admission_source,primary_diag,secondary_diag,additional_diag
0,5.0,1,41,0,1,0,0,0,1,0,0,0,Caucasian,Female,Other,Unknown,Other/Unknown,Referral/Clinic,Diabetes,Unknown,Unknown
1,15.0,3,59,0,18,0,0,0,9,0,1,0,Caucasian,Female,Missing,Emergency,Home,Emergency Room,Other,Diabetes,Other
2,25.0,2,11,5,13,2,0,1,6,3,1,0,AfricanAmerican,Female,Missing,Emergency,Home,Emergency Room,Pregnancy,Diabetes,Other
3,35.0,2,44,1,16,0,0,0,7,0,1,0,Caucasian,Male,Missing,Emergency,Home,Emergency Room,Infectious,Diabetes,Circulatory
4,45.0,1,51,0,8,0,0,0,5,0,2,0,Caucasian,Male,Missing,Emergency,Home,Emergency Room,Neoplasms,Neoplasms,Diabetes


In [12]:
print('train:', data['X_train'].shape, ' test:', data['X_test'].shape)
data['y_train'].value_counts(normalize=True).rename('train').to_frame().join(data['y_test'].value_counts(normalize=True).rename('test'))

train: (81412, 21)  test: (20354, 21)


,train,test
readmitted_30d,,
0,0.888395,0.888425
1,0.111605,0.111575


## 5 · Modelling

Four model families are compared on the same stratified 80/20 split (`random_state=42`):

1. **Logistic Regression** — interpretable linear baseline (class-weighted)
2. **Random Forest** — bagged trees (class-weighted)
3. **XGBoost** — gradient boosting (`scale_pos_weight` ≈ 8 for imbalance)
4. **Keras MLP** — 256→128→64 neural network with dropout + early stopping
5. **SMOTE + LogReg** — oversampling alternative to class weights

Hyperparameters for the Random Forest and XGBoost were selected with a small `RandomizedSearchCV` (3-fold, ROC-AUC) in `src/train.py`; this notebook uses the winning settings so it runs in minutes.

In [13]:
results = train_all_models(data, tune=False)

[1/5] Logistic Regression ...


  logreg           ROC-AUC 0.6684 | PR-AUC 0.2077 | F1 0.2703 | fit   3.5s


[2/5] Random Forest (stored best params) ...


  random_forest    ROC-AUC 0.6777 | PR-AUC 0.2167 | F1 0.2781 | fit  14.0s


[3/5] XGBoost (stored best params) ...


  xgboost          ROC-AUC 0.6709 | PR-AUC 0.2154 | F1 0.2727 | fit   3.7s


[4/5] Keras MLP neural network ...


  keras_mlp        ROC-AUC 0.6714 | PR-AUC 0.2116 | F1 0.2658 | fit  13.7s | epochs 9


[5/5] SMOTE + Logistic Regression ...


  logreg_smote     ROC-AUC 0.6535 | PR-AUC 0.1996 | F1 0.2585 | fit   6.5s


## 6 · Evaluation

In [14]:
rows = {n: results[n]['metrics'] for n in results if not n.startswith('_')}
summary = pd.DataFrame(rows).T[['roc_auc', 'pr_auc', 'accuracy', 'precision', 'recall', 'f1', 'f1_best_threshold']]
summary.sort_values('roc_auc', ascending=False).style.format('{:.4f}').background_gradient(subset=['roc_auc', 'pr_auc'], cmap='Greens')

,roc_auc,pr_auc,accuracy,precision,recall,f1,f1_best_threshold
random_forest,0.6777,0.2167,0.6780,0.1854,0.5557,0.2781,0.2784
keras_mlp,0.6714,0.2116,0.5851,0.1656,0.6733,0.2658,0.2719
xgboost,0.6709,0.2154,0.6570,0.1786,0.5764,0.2727,0.2740
logreg,0.6684,0.2077,0.6387,0.1745,0.5997,0.2703,0.2746
logreg_smote,0.6535,0.1996,0.6284,0.1662,0.5804,0.2585,0.2670


### 6.1 ROC and Precision-Recall curves

In [15]:
y_test = data['y_test'].values
probas = {n: results[n]['proba_test'] for n in results if not n.startswith('_')}
fig, ax = plt.subplots(1, 2, figsize=(13, 5.5))
plt.sca(ax[0]); ev.plot_roc_curves(y_test, probas)
plt.sca(ax[1]); ev.plot_pr_curves(y_test, probas)
plt.tight_layout(); plt.show()

The **PR curve** is the honest view under an 11% positive rate: precision collapses as we push for high recall, so deployment threshold must be chosen per hospital priorities (missed patients vs. intervention capacity).

### 6.2 Confusion matrices (0.5 threshold)

In [16]:
fig = ev.plot_confusion_matrices(y_test, probas)
plt.show()

### 6.3 What the model looks at — XGBoost feature importance

In [17]:
fig = ev.plot_feature_importance(results['xgboost']['model'])
plt.show()

**Prior utilisation** (`number_inpatient`, `service_utilization`), **discharge disposition** and **diagnosis mix** dominate — exactly what the clinical literature reports for diabetic readmission.

### 6.4 Neural network training dynamics

In [18]:
fig = ev.plot_keras_history(results['keras_mlp']['history'])
plt.show()

## 7 · Conclusions

- Tree ensembles (XGBoost / Random Forest) beat the linear baseline and the neural network on this tabular, wide-but-shallow dataset — expected: ~100k rows of mostly categorical clinical features.
- Class weighting handles the 11% positive rate at least as well as SMOTE, with far less compute.
- With ROC-AUC around **0.69–0.70** the model is far better than chance (0.50) and in line with published results on this dataset; readmission is genuinely hard to predict from administrative data alone.
- Next steps: richer feature engineering from medication changes, calibration analysis, cost-sensitive thresholds, and fairness checks across race/age groups.

---
*Reproduce: `python src/download_data.py && python src/train.py` — a live demo is available via `streamlit run app/app.py`.*